In [ ]:
import os
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(name, '1')

from math import ceil
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.special import logsumexp, xlogy

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#CDD3D8', 'axes.linewidth': .8,
                     'axes.labelcolor': '#33414C', 'text.color': '#33414C',
                     'xtick.color': '#6B7680', 'ytick.color': '#6B7680',
                     'axes.titlesize': 11, 'axes.titleweight': 'medium',
                     'lines.linewidth': 1.7, 'xtick.labelsize': 9, 'ytick.labelsize': 9})


In [ ]:
# Parameters
calibration_seed = 90261020
device_seed = 100261020
cells = 32
edge_cells = 2
uniform_loss = 0.015
readout_visibility = 0.96
r_grid = np.linspace(.5, 1.7, 25)
delta_grid = [-0.1, -0.06, -0.03, -0.015, 0.0, 0.015, 0.03, 0.06, 0.1]
mass_grid = [-0.16, -0.1, -0.06, -0.03, -0.015, 0.0, 0.015, 0.03, 0.06, 0.1, 0.16]
momenta = [0, np.pi / 2, np.pi - .5, np.pi - .18, np.pi, np.pi + .18, np.pi + .5]
times = [0.6, 1.6, 4.0, 8.0]
axis_pairs = [[2, 2], [0, 1], [1, 2], [2, 0]]
edge_times = [8.0, 12.0, 20.0]
gap_compatible_max = 0.04
gap_resolved_min = 0.12
ordinary_r_max = 0.85
topological_r_min = 1.15
posterior_threshold = 0.95
minimum_rounds = 8
stable_looks = 2
rounds = 24
shots_per_round = 64
calibration_devices = 64
false_alarm_target = 0.05
persistence = 2
covariance_ridge = 1.0
devices_per_case = 12
noise_repeats = 2
methods = ['fixed_schedule', 'parameter_information', 'class_information']
cases = ['ordinary_compatible', 'topological_compatible', 'resolved_gapped', 'small_gap_gray', 'multicritical_neighborhood', 'outside_model']
CLASS_NAMES = ('ordinary_compatible', 'topological_compatible', 'resolved_gapped', 'unresolved')
AXES = np.eye(3)
METHOD_NAMES = {'fixed_schedule': 'Fixed schedule', 'parameter_information': 'Parameter information', 'class_information': 'Physical-class information'}


In [ ]:
# Hamiltonian and measurement probabilities
def model_labels(theta):
    theta = np.atleast_2d(theta)
    r, d, m = theta[:, :3].T
    a, b, c = (1 + d, 1 + r, r)
    x = np.clip(-b * (a + c) / (4 * a * c), -1, 1)
    z = x + 1j * np.sqrt(np.maximum(0, 1 - x * x))
    gaps = 2 * np.sqrt(abs(a + b * z + c * z * z) ** 2 + m * m)
    labels = np.full(len(theta), 3, int)
    labels[gaps >= gap_resolved_min - 1e-12] = 2
    compatible = gaps <= gap_compatible_max + 1e-12
    labels[compatible & (r <= ordinary_r_max + 1e-12)] = 0
    labels[compatible & (r >= topological_r_min - 1e-12)] = 1
    return (labels, gaps)


def action_catalog():
    actions = []
    for k in momenta:
        for time in times:
            for prep, read in axis_pairs:
                actions.append({'kind': 'bulk', 'k': k, 'time': time, 'prep': prep, 'read': read})
    actions.extend(({'kind': 'boundary', 'time': t} for t in edge_times))

    def get(k, time, prep, read):
        return next((i for i, a in enumerate(actions) if a['kind'] == 'bulk' and abs(a['k'] - k) < 1e-10 and (a['time'] == time) and (a['prep'] == prep) and (a['read'] == read)))
    pi = np.pi
    initial = [get(pi, 4.0, 0, 1), get(pi, 4.0, 1, 2), get(pi - 0.5, 0.6, 2, 2), get(pi + 0.5, 1.6, 2, 0), next((i for i, a in enumerate(actions) if a['kind'] == 'boundary')), get(0.0, 0.6, 2, 2)]
    fixed = [get(pi, 8.0, 0, 1), get(pi, 8.0, 1, 2), get(pi - 0.18, 4.0, 2, 2), get(pi + 0.18, 4.0, 2, 0), get(pi - 0.5, 1.6, 2, 2), get(pi + 0.5, 1.6, 0, 1), get(np.pi / 2, 0.6, 1, 2), next((i for i, a in enumerate(actions) if a['kind'] == 'boundary' and a['time'] == 12.0))]
    return (actions, initial, fixed)


def bulk_probabilities(theta, actions, third_hop=0.0):
    theta = np.atleast_2d(theta)
    r, d, m = theta[:, :3].T
    k = np.asarray([a['k'] for a in actions])
    t = np.asarray([a['time'] for a in actions])
    z = np.exp(1j * k)
    q = (1 + z)[None, :] * (1 + r[:, None] * z[None, :]) + d[:, None] + third_hop * z[None, :] ** 3
    h = np.stack([q.real, q.imag, np.broadcast_to(m[:, None], q.shape)], axis=-1)
    energy = np.linalg.norm(h, axis=-1)
    axis = np.divide(h, energy[:, :, None], out=np.zeros_like(h), where=energy[:, :, None] > 1e-15)
    prep = AXES[[a['prep'] for a in actions]]
    read = AXES[[a['read'] for a in actions]]
    angle = 2 * energy * t[None, :]
    c, s = (np.cos(angle)[:, :, None], np.sin(angle)[:, :, None])
    dot = np.sum(axis * prep[None, :, :], axis=-1)[:, :, None]
    evolved = prep[None, :, :] * c + np.cross(axis, prep[None, :, :]) * s + axis * dot * (1 - c)
    signal = np.sum(evolved * read[None, :, :], axis=-1)
    plus = 0.5 + 0.5 * readout_visibility * signal
    survival = np.exp(-uniform_loss * t)
    return np.stack([survival[None, :] * plus, survival[None, :] * (1 - plus), np.broadcast_to(1 - survival, plus.shape)], axis=-1)


def edge_signals(cells, edge_cells, r, d, m, third_hop, times):
    h = np.diag(np.tile([m, -m], cells)).astype(float)
    for n in range(cells):
        for shift, hop in enumerate([1 + d, 1 + r, r, third_hop]):
            if n >= shift:
                a, b = (2 * n, 2 * (n - shift) + 1)
                h[a, b] = h[b, a] = hop
    e, v = eigh(h, driver='evd')
    amplitude = v[:2 * edge_cells] * v[0][None, :] @ np.exp(-1j * e[:, None] * np.asarray(times)[None, :])
    return np.sum(abs(amplitude) ** 2, axis=0)


def probability_bank(theta, actions, third_hop=0.0):
    theta = np.atleast_2d(theta)
    bulk = [i for i, a in enumerate(actions) if a['kind'] == 'bulk']
    edge = [i for i, a in enumerate(actions) if a['kind'] == 'boundary']
    result = np.empty((len(theta), len(actions), 3))
    result[:, bulk] = bulk_probabilities(theta, [actions[i] for i in bulk], third_hop)
    times = tuple((actions[i]['time'] for i in edge))
    for j, (r, d, m) in enumerate(theta):
        signals = edge_signals(cells, edge_cells, float(r), float(d), float(m), float(third_hop), times)
        plus = 0.5 + readout_visibility * (signals - 0.5)
        survival = np.exp(-uniform_loss * np.asarray(times))
        result[j, edge] = np.column_stack([survival * plus, survival * (1 - plus), 1 - survival])
    return np.clip(result, 0, 1)

actions, initial, fixed = action_catalog()
theta = np.array([(r, d, m) for r in r_grid for d in delta_grid for m in mass_grid])
particle_labels, gaps = model_labels(theta)
bank = probability_bank(theta, actions)
class_mask = (np.arange(4)[:, None] == particle_labels[None, :]).astype(float)
conditional_entropy = -np.sum(xlogy(bank, bank), axis=-1)


In [ ]:
# Information gain and predictive mismatch
def information(weights, method):
    mean = np.einsum('p,pay->ay', weights, bank)
    entropy = -np.sum(xlogy(mean, mean), axis=-1)
    if method == 'parameter_information':
        return np.maximum(0, entropy - weights @ conditional_entropy)
    class_weights = class_mask * weights[None, :]
    masses = class_weights.sum(axis=1)
    joint = (class_weights @ bank.reshape(len(weights), -1)).reshape(4, len(actions), 3)
    conditional = np.divide(joint, masses[:, None, None], out=np.zeros_like(joint), where=masses[:, None, None] > 1e-300)
    return np.maximum(0, entropy - np.einsum('c,cay->a', masses, -xlogy(conditional, conditional)))


def mismatch(weights, action, counts):
    counts = counts.astype(float)
    p = bank[:, action]
    mean = weights @ p
    second = np.einsum('p,pi,pj->ij', weights, p, p)
    n = counts.sum()
    covariance = n * (np.diag(mean) - second) + n * n * (second - np.outer(mean, mean))
    covariance = (covariance + covariance.T) / 2
    residual = counts[:2] - n * mean[:2]
    score = residual @ np.linalg.solve(covariance[:2, :2] + covariance_ridge * np.eye(2), residual)
    return float(max(0, score))


In [ ]:
# Sequential Bayesian measurements
def identify(actual, method, threshold, seed, full=False):
    rng = np.random.default_rng(seed)
    log_weights = np.full(len(theta), -np.log(len(theta)))
    scores, trace = [], []
    previous, alarm_round = None, None
    stable, label, maximum = 0, 3, 0.
    boundary_seen = False
    for step in range(rounds):
        if step < len(initial):
            action = initial[step]
        elif method == 'fixed_schedule':
            action = fixed[(step - len(initial)) % len(fixed)]
        else:
            action = int(np.argmax(information(np.exp(log_weights), method)))
        counts = rng.multinomial(shots_per_round, actual[action])
        scores.append(mismatch(np.exp(log_weights), action, counts))
        sustained = min(scores[-persistence:]) if len(scores) >= persistence else 0.
        maximum = max(maximum, sustained)
        if sustained > threshold and alarm_round is None:
            alarm_round = step + 1
        likelihood = np.log(np.maximum(bank[:, action], 1e-300)) @ counts
        log_weights += likelihood - logsumexp(log_weights + likelihood)
        class_probs = class_mask @ np.exp(log_weights)
        top = int(np.argmax(class_probs))
        confident = class_probs[top] >= posterior_threshold
        stable = stable + 1 if top == previous and confident else int(confident)
        previous = top
        boundary_seen |= actions[action]['kind'] == 'boundary'
        trace.append((action, class_probs.copy(), sustained))
        if (step + 1 >= minimum_rounds and stable >= stable_looks and
                (top != 1 or boundary_seen) and alarm_round is None and not full):
            label = top
            break
    return dict(method=method, label=label, shots=(step + 1) * shots_per_round,
                alarm_round=alarm_round, max_sustained_score=maximum, trace=trace)


In [ ]:
# Simulated device families
def draw_device(rng, case):
    third = 0.0
    if case == 'ordinary_compatible':
        theta = [rng.uniform(0.55, 0.78), rng.uniform(-0.003, 0.003), rng.uniform(-0.003, 0.003)]
    elif case == 'topological_compatible':
        theta = [rng.uniform(1.3, 1.65), rng.uniform(-0.003, 0.003), rng.uniform(-0.003, 0.003)]
    elif case == 'resolved_gapped':
        theta = [rng.uniform(0.55, 1.65), rng.uniform(-0.04, 0.04), rng.choice([-1, 1]) * rng.uniform(0.09, 0.14)]
    elif case == 'small_gap_gray':
        theta = [rng.uniform(1.3, 1.6), rng.uniform(-0.003, 0.003), rng.choice([-1, 1]) * rng.uniform(0.025, 0.045)]
    elif case == 'multicritical_neighborhood':
        theta = [rng.uniform(0.97, 1.03), rng.uniform(-0.003, 0.003), rng.uniform(-0.003, 0.003)]
    elif case == 'outside_model':
        theta = [rng.uniform(0.6, 1.6), rng.uniform(-0.01, 0.01), rng.uniform(-0.01, 0.01)]
        third = float(rng.choice([-1, 1]) * rng.uniform(0.14, 0.22))
    return (list(map(float, theta)), third)


def reference_class(theta, third):
    if third:
        k = np.linspace(0, 2 * np.pi, 32769)
        z = np.exp(1j * k)
        r, d, m = theta
        q = (1 + z) * (1 + r * z) + d + third * z ** 3
        gap = 2 * np.min(np.sqrt(abs(q) ** 2 + m * m))
        return 2 if gap >= gap_resolved_min else 3
    return int(model_labels(theta)[0][0])


In [ ]:
# Calibration and device measurements
calibration = {method: [] for method in methods}
rng = np.random.default_rng(calibration_seed)
for device in range(calibration_devices):
    case = str(rng.choice(cases[:-1]))
    device_theta, third = draw_device(rng, case)
    actual = probability_bank(device_theta, actions, third)[0]
    for method in methods:
        result = identify(actual, method, float('inf'), calibration_seed + 100000 + device, full=True)
        calibration[method].append(result['max_sustained_score'])
rank = ceil((calibration_devices + 1) * (1 - false_alarm_target))
thresholds = {method: np.sort(values)[rank - 1] for method, values in calibration.items()}

trials = []
for case_index, case in enumerate(cases):
    rng = np.random.default_rng(device_seed + case_index * 10000)
    for device in range(devices_per_case):
        device_theta, third = draw_device(rng, case)
        actual = probability_bank(device_theta, actions, third)[0]
        group = []
        for noise in range(noise_repeats):
            for method in methods:
                seed = device_seed + 100000 + case_index * 10000 + device * 100 + noise
                result = identify(actual, method, thresholds[method], seed)
                group.append(result)
                if case == 'topological_compatible' and device == 0 and noise == 0 and method == 'class_information':
                    example = result
        truth = reference_class(device_theta, third)
        for result in group:
            result.update(case=case, truth=truth)
        trials.extend(group)


In [ ]:
# Results
results = {}
print('Policy                       Definitive  Correct  Mean shots')
for method in methods:
    inside = [r for r in trials if r['method'] == method and r['case'] != 'outside_model']
    outside = [r for r in trials if r['method'] == method and r['case'] == 'outside_model']
    matrix = np.zeros((4, 4), int)
    np.add.at(matrix, ([r['truth'] for r in inside], [r['label'] for r in inside]), 1)
    definitive = matrix[:, :3].sum()
    correct = np.trace(matrix[:3, :3])
    mean_shots = np.mean([r['shots'] for r in inside])
    results[method] = dict(coverage=definitive / len(inside), accuracy=correct / definitive if definitive else np.nan,
                           mean_shots=mean_shots, confusion=matrix,
                           alarm_rate=np.mean([r['alarm_round'] is not None for r in outside]))
    print(f'{METHOD_NAMES[method]:28} {definitive:3}/{len(inside):<3}     {correct:3}/{definitive:<3}  {mean_shots:10.1f}')
print(f'Mean shot reduction: {100 * (1 - results["class_information"]["mean_shots"] / results["fixed_schedule"]["mean_shots"]):.1f}%')
print(f'Calibration shots: {len(methods) * calibration_devices * rounds * shots_per_round}')
print('Outside-model alarm rates: ' + ', '.join(f'{METHOD_NAMES[m]} {r["alarm_rate"]:.1%}' for m, r in results.items()))


In [ ]:
# Figures
names = {'fixed_schedule': 'Fixed', 'parameter_information': 'Parameter info.', 'class_information': 'Class info.'}
policy_colors = {'fixed_schedule': '#A4ADB5', 'parameter_information': '#537F9B', 'class_information': '#287D72'}
positions = np.arange(len(methods))
fig, axes = plt.subplots(1, 3, figsize=(9.8, 2.8), sharey=True, layout='constrained')
panels = [('coverage', '(a) Definitive answers'),
          ('accuracy', '(b) Definitive accuracy'),
          ('mean_shots', '(c) Mean shots')]
for ax, (key, title) in zip(axes, panels):
    numbers = [results[m][key] for m in methods]
    ax.scatter([n or 0 for n in numbers], positions, s=46, c=[policy_colors[m] for m in methods], zorder=3)
    for row, number in enumerate(numbers):
        label = 'n/a' if np.isnan(number) else f'{number:.0f}' if key.endswith('shots') else f'{100 * number:.1f}%'
        ax.annotate(label, (number or 0, row), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9)
    if key.endswith('shots'):
        ax.set_xlim(0, (rounds * shots_per_round) * 1.05)
        ax.set_xticks([0, (rounds * shots_per_round) / 2, (rounds * shots_per_round)])
    else:
        ax.set_xlim(0, 1.08)
        ax.set_xticks([0, .5, 1], ['0%', '50%', '100%'])
    ax.set_title(title, loc='left')
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', color='#E8ECEF', linewidth=.7)
    ax.set_axisbelow(True)
axes[0].set_yticks(positions, [names[m] for m in methods])
axes[0].set_ylim(len(methods) - .5, -.65)
plt.show()
plt.close(fig)

trace = example['trace']
shot_axis = shots_per_round * np.arange(1, len(trace) + 1)
class_labels = ['Ordinary', 'Topological', 'Gapped', 'Unresolved']
colors = ['#537F9B', '#287D72', '#B47A45', '#9AA2AA']
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), layout='constrained')
for i, (name, color, style) in enumerate(zip(class_labels, colors, ['-', '-', ':', '--'])):
    axes[0].plot(shot_axis, [r[1][i] for r in trace],
                 label=name, color=color, linestyle=style)
axes[0].axhline(posterior_threshold, color='#B8BEC4', linestyle='--', linewidth=.9)
axes[0].text(.02, posterior_threshold, 'Stop level', transform=axes[0].get_yaxis_transform(),
             ha='left', va='bottom', fontsize=8, color='#747E87')
axes[0].set(xlabel='Shots', ylabel='Class probability', ylim=(-.03, 1.06), yticks=[0, .5, 1])
for batch, row in enumerate(trace, 1):
    action = actions[row[0]]
    bulk = action['kind'] == 'bulk'
    axes[1].scatter(batch, action.get('k', -.5), s=18 + 3 * action['time'],
                    color='#537F9B' if bulk else '#B47A45', marker='o' if bulk else 's', zorder=3)
axes[1].axvspan(.5, len(initial) + .5, color='#F0F3F5', zorder=0)
axes[1].axvline(len(initial) + .5, linestyle=':', color='#A4ADB5', linewidth=1)
axes[1].set_yticks([-.5, 0, np.pi / 2, np.pi], ['Edge', '0', 'π/2', 'π'])
axes[1].set(xlabel='Batch (size: time)', ylabel='Momentum')
axes[2].plot(np.arange(1, len(trace) + 1), [r[2] for r in trace],
             color='#537F9B', marker='o', markersize=3)
axes[2].axhline(thresholds['class_information'], color='#B47A45', linestyle='--', linewidth=1)
axes[2].text(.98, thresholds['class_information'], 'Alarm', transform=axes[2].get_yaxis_transform(),
             ha='right', va='bottom', fontsize=8, color='#946238')
axes[2].set(xlabel='Batch', ylabel='Mismatch score')
for ax, title in zip(axes, ['(a) Evidence', '(b) Measurements', '(c) Model check']):
    ax.set_title(title, loc='left')
    ax.locator_params(axis='x', nbins=4)
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=4, frameon=False, fontsize=9)
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, len(methods), figsize=(10, 3.4), sharey=True, layout='constrained', squeeze=False)
rows = ['Ordinary', 'Topological', 'Gapped', 'Unresolved']
columns = ['Ordinary', 'Topo.', 'Gapped', 'Inconcl.']
for index, (ax, method) in enumerate(zip(axes[0], methods)):
    matrix = results[method]['confusion']
    normalized = matrix / np.maximum(1, matrix.sum(axis=1))[:, None]
    image = ax.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            if matrix[i, j]:
                ax.text(j, i, str(matrix[i, j]), ha='center', va='center',
                        color='white' if normalized[i, j] > .55 else '#33414C', fontsize=10)
    ax.set_xticks(range(4), columns, fontsize=9)
    ax.set_yticks(range(4), rows)
    ax.set_xticks(np.arange(5) - .5, minor=True)
    ax.set_yticks(np.arange(5) - .5, minor=True)
    ax.grid(which='minor', color='white', linewidth=1.5)
    ax.tick_params(which='both', length=0)
    ax.set_title(f'({chr(97 + index)}) {names[method]}', loc='left')
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.supxlabel('Controller decision', fontsize=10)
fig.supylabel('Reference class', fontsize=10)
colorbar = fig.colorbar(image, ax=axes[0].tolist(), shrink=.75, fraction=.025, pad=.02, ticks=[0, .5, 1])
colorbar.ax.set_yticklabels(['0%', '50%', '100%'])
colorbar.set_label('Row share', fontsize=9)
colorbar.outline.set_visible(False)
plt.show()
plt.close(fig)
